# DOCUMENTATION COMPLETE IA CNN U-NET

La première étape, une fois les images originales récupérées, consiste à les rogner afin de les rendre carrées et d’éliminer les zones inutiles (fonds, bords, éléments parasites, etc.). Cette étape permet d’homogénéiser le format des images pour l’apprentissage du modèle et d’améliorer la qualité des données en supprimant les informations non pertinentes.

Pour effectuer ce rognage, utilisez le script suivant.

### 🔹Étape 1 – Rogner les images du dataset

Cette fonction permet de rogner automatiquement toutes les images d’un dossier en fonction de coordonnées définies manuellement, afin de ne conserver que la partie utile (par exemple : la carcasse) et de rendre les images plus homogènes.

✅ Objectifs :
Supprimer les zones inutiles (fonds, bords, marquages…)

Préparer les images pour la normalisation et l’entraînement du modèle

📌 Comment utiliser ce script :
Spécifiez le chemin du dossier contenant vos images brutes (input_folder)

Choisissez un dossier de sortie pour stocker les images rognées (output_folder)

In [ ]:
import os
from PIL import Image

def crop_images(input_folder, output_folder, crop_coordinates):
    """
    Rognage par lot des images dans un dossier donné.

    Paramètres :
    - input_folder : str, chemin vers le dossier contenant les images d'origine
    - output_folder : str, chemin vers le dossier de sortie pour les images rognées
    - crop_coordinates : tuple (left, top, right, bottom), coordonnées du rectangle de rognage en pixels

    Résultat :
    - Sauvegarde les images rognées dans output_folder avec les mêmes noms de fichiers
    """
    
    # Créer le dossier de sortie s'il n'existe pas
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Parcourir toutes les images du dossier source
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):  # Vérifie les extensions d'image
            img_path = os.path.join(input_folder, filename)  # Chemin complet de l'image
            img = Image.open(img_path)  # Ouvrir l'image avec PIL

            # Rogner l'image avec les coordonnées données
            cropped_img = img.crop(crop_coordinates)

            # Sauvegarder l'image rognée dans le dossier de sortie
            output_path = os.path.join(output_folder, filename)
            cropped_img.save(output_path)

            print(f'✅ Image \"{filename}\" rognée et sauvegardée dans : {output_path}')

# Exemple d'utilisation :
input_folder = './CHEMIN_VERS_VOS_IMAGES'               # Dossier contenant les images brutes
output_folder = './CHEMIN_DOSSIER_FINAL'         # Dossier où seront stockées les images rognées
crop_coordinates = (477, 0, 1773, 1296)      # Coordonnées de la zone à conserver (pixels)

crop_images(input_folder, output_folder, crop_coordinates)


### 🔹 Étape 2 – Annotation des images avec Labelbox

Une fois les images rognées, il est nécessaire de les annoter manuellement avant de les utiliser pour entraîner un modèle de segmentation.
L’outil utilisé ici est Labelbox, une plateforme en ligne permettant d’annoter facilement les images et d’exporter les masques.

🎯 Objectif :
Annoter les zones d’intérêt sur les images (gras ou canal médullaire)

Exporter les masques au format utilisable pour l’entraînement U-Net

📝 Procédure à suivre :
1. Créer un projet sur Labelbox :
Aller sur Labelbox

Créer un nouveau projet

Choisir un nom clair (ex : ANNOTATION_GLUTEUS_AB13)

Dans la rubrique Ontology, ajouter une seule classe :

GLUTEUS ou CANAL_MEDULAIRE

⚠️ Ne pas annoter les deux simultanément : les fusions seront faites plus tard

2. Importer les images :
Depuis l’onglet Dataset, importer les images rognées (issues de l’étape 1)

Laissez-les charger, puis assignez-les au projet

3. Annoter toutes les images :
Utiliser l’outil de dessin polygonal ou pinceau

Annoter uniquement une classe par projet

Vérifier que chaque image a bien une annotation

4. Générer une clé API :
Aller dans Account settings > API Keys

Créer une clé d’API avec les droits nécessaires et la copier

5. Exporter les annotations :
Une fois toutes les images annotées, lancer un export dans Labelbox

Récupérer l'ID de la tâche d'export et utiliser le code ci-dessous pour récupérer les fichiers au format JSON et les télécharger localement

#### 🔑 Téléchargement automatique via API Labelbox

In [ ]:
# --- Connexion à l'API Labelbox et export JSON ---
import labelbox
import json

# ⚠️ Remplacez par votre vraie API Key
LB_API_KEY = 'API_KEY'
EXPORT_TASK_ID = 'ID_EXPORT'  # Copiez ici l'ID obtenu après export dans Labelbox

client = labelbox.Client(api_key=LB_API_KEY)
export_task = labelbox.ExportTask.get_task(client, EXPORT_TASK_ID)

# Optionnel : afficher les objets JSON exportés en flux
def json_stream_handler(output: labelbox.BufferedJsonConverterOutput):
    print(output.json)

# Obtenir les données JSON dans un fichier local
export_task.get_buffered_stream(stream_type=labelbox.StreamType.RESULT).start(stream_handler=json_stream_handler)
export_json = [data_row.json for data_row in export_task.get_buffered_stream()]

# Sauvegarde locale
with open("LABELBOX.json", "w", encoding="utf-8") as f:
    json.dump(export_json, f, ensure_ascii=False, indent=4)

print("✅ Export JSON sauvegardé dans LABELBOX.json")


#### 📥 Téléchargement des images et masques depuis le JSON Labelbox


In [ ]:
import os
import requests
import json

# === Paramètres ===
API_TOKEN = "API_KEY"  # Token Labelbox
json_file = "./LABELBOX.json"  # Fichier généré précédemment
images_dir = "./MP_IMG_CANAL"  # Dossier de destination des images originales
masks_dir = "./MP_MSK_CANAL"   # Dossier de destination des masques

# Créer les dossiers si nécessaire
os.makedirs(images_dir, exist_ok=True)
os.makedirs(masks_dir, exist_ok=True)

def download_file(url, save_path, headers=None):
    """Télécharge un fichier depuis une URL avec gestion des erreurs."""
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        with open(save_path, "wb") as f:
            f.write(response.content)
        print(f"✅ Téléchargé : {save_path}")
    except Exception as e:
        print(f"❌ Erreur : {url} - {e}")

def main():
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    for item in data:
        mask_url = None
        projects = item.get("projects", {})
        
        # Recherche du masque dans la structure JSON
        for project in projects.values():
            labels = project.get("labels", [])
            for label in labels:
                annotations = label.get("annotations", {})
                for obj in annotations.get("objects", []):
                    mask = obj.get("mask", {})
                    mask_url = mask.get("url")
                    if mask_url:
                        break
                if mask_url:
                    break
            if mask_url:
                break

        if not mask_url:
            print("❗ Aucun mask trouvé, image ignorée.")
            continue

        # Récupération des infos sur l'image d'origine
        data_row = item.get("data_row", {})
        external_id = data_row.get("external_id")
        row_data_url = data_row.get("row_data")

        if external_id and row_data_url:
            # Télécharger l’image
            download_file(row_data_url, os.path.join(images_dir, external_id))
        else:
            print("⚠️ Image d'origine introuvable pour un item.")
            continue

        # Télécharger le masque (nommé mask_XXX.jpg)
        mask_filename = "mask_" + external_id
        headers = {"Authorization": f"Bearer {API_TOKEN}"}
        download_file(mask_url, os.path.join(masks_dir, mask_filename), headers=headers)

if __name__ == "__main__":
    main()


### 🔹 Étape 3 – Redimensionnement des images

Une fois les IMAGES et MASQUES rognées et annotées, il est nécessaire de les redimensionner pour les adapter à l'entrée du modèle U-Net. Les modèles de deep learning exigent en général que toutes les images aient la même taille (souvent une puissance de 2, comme 256, 512, 1024, etc.).

Ce script permet de redimensionner par lot toutes les images d’un dossier, tout en conservant le rapport hauteur/largeur si une seule dimension est précisée.
Il permet aussi de renommer les fichiers dans l’ordre (0001.jpg, 0002.jpg, etc.), ce qui peut faciliter leur gestion ou annotation ultérieure.

📌 Comment utiliser ce script :
Définir le dossier source contenant les images à redimensionner (folder_path)

Spécifier un dossier de sortie (output_folder)

Choisir les dimensions souhaitées (new_width, new_height)

Si une seule dimension est précisée, l’autre est calculée pour conserver le ratio d’aspect

Lancer le script pour générer les images redimensionnées

In [ ]:
import os
import cv2
import time

# === Configuration ===
folder_path = "./A_ANNOTER/MP_MSK_DOUBLE_1296"    # Dossier d'origine
output_folder = "./A_ANNOTER/MP_MSK_DOUBLE_512"   # Dossier pour les images redimensionnées
new_width = 512                                   # Largeur cible (hauteur auto si non précisée)

start_time = time.time()  # Pour mesurer le temps total d'exécution

# === Fonction de redimensionnement d'une image ===
def resize_image(image, new_width=None, new_height=None):
    """
    Redimensionne une image en conservant le ratio si une seule dimension est spécifiée.
    """
    if new_width is None and new_height is None:
        return image  # Aucun redimensionnement
    
    # Calcul du ratio pour conserver les proportions
    if new_width is None:
        ratio = new_height / image.shape[0]
        new_width = int(image.shape[1] * ratio)
    elif new_height is None:
        ratio = new_width / image.shape[1]
        new_height = int(image.shape[0] * ratio)
    
    # Redimensionnement avec interpolation
    resized_image = cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)
    return resized_image

# === Fonction de traitement d’un dossier complet ===
def resize_images_in_folder(folder_path, output_folder, new_width=None, new_height=None):
    """
    Redimensionne toutes les images d'un dossier et les enregistre dans un nouveau dossier.
    Les images sont renommées en séquence (0001.jpg, 0002.jpg...).
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    numero_photo = 1  # Pour renommer les images

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            # Chemin vers l'image originale
            image_path = os.path.join(folder_path, filename)
            image = cv2.imread(image_path)

            # Redimensionner
            resized_image = resize_image(image, new_width, new_height)

            # Nouveau nom : 0001.jpg, 0002.jpg, etc.
            extension = os.path.splitext(filename)[1]
            nouveau_nom = str(numero_photo).zfill(4) + extension
            numero_photo += 1

            # Enregistrer l’image dans le dossier de sortie
            output_path = os.path.join(output_folder, nouveau_nom)
            cv2.imwrite(output_path, resized_image)

            print(f"✅ Image {filename} redimensionnée et sauvegardée sous {nouveau_nom}")

# === Exécution ===
resize_images_in_folder(folder_path, output_folder, new_width=new_width)

print("⏱️ Temps d'exécution total : %.2f secondes" % (time.time() - start_time))


### 🔹 Étape 4 – Fusion des masques annotés : GLUTEUS + CANAL_MÉDULAIRE


Après avoir exporté séparément les annotations des deux zones d’intérêt (GLUTEUS et CANAL_MÉDULAIRE) depuis Labelbox, il est nécessaire de fusionner les masques pour créer une carte d’annotation unique multiclasse.

🎯 Objectif :
Créer des masques d'entraînement avec 3 classes :

0 : fond (aucune annotation)

1 : GLUTEUS

2 : CANAL_MÉDULAIRE

Résoudre automatiquement les éventuels recouvrements entre masques

📌 Conditions :
Les fichiers de masques doivent avoir le même nom dans les deux dossiers (mask_IMG001.png, etc.)

Les masques doivent être au format PNG en niveaux de gris (valeurs 0 ou 255)

Chaque masque contient une seule classe (0 = fond, 255 = zone annotée)

In [ ]:
import os
from PIL import Image
import numpy as np

# === CONFIGURATION ===
mask_dir1 = './A_ANNOTER/MSK'           # Masques GLUTEUS (label 1)
mask_dir2 = './A_ANNOTER/MSK_CANAL'     # Masques CANAL_MEDULAIRE (label 2)
output_dir = './A_ANNOTER/MSK_DOUBLE'   # Dossier pour les masques fusionnés

# Créer le dossier de sortie s'il n'existe pas
os.makedirs(output_dir, exist_ok=True)

# Identifier les fichiers communs dans les deux dossiers
files1 = {f for f in os.listdir(mask_dir1) if f.lower().endswith('.png')}
files2 = {f for f in os.listdir(mask_dir2) if f.lower().endswith('.png')}
common_files = sorted(files1 & files2)

print(f"📦 Nombre de masques à fusionner : {len(common_files)}")

for fname in common_files:
    # Charger les deux masques en niveau de gris (L)
    m1 = np.array(Image.open(os.path.join(mask_dir1, fname)).convert('L'))  # GLUTEUS
    m2 = np.array(Image.open(os.path.join(mask_dir2, fname)).convert('L'))  # CANAL

    # Binariser : 0 (fond), 1 (zone annotée)
    m1 = (m1 > 128).astype(np.uint8)
    m2 = (m2 > 128).astype(np.uint8)

    # Masque fusionné (valeurs : 0 = fond, 1 = GLUTEUS, 2 = CANAL)
    fused = np.zeros_like(m1, dtype=np.uint8)
    fused[m1 == 1] = 1               # Classe 1 : GLUTEUS
    fused[m2 == 1] = 2               # Classe 2 : CANAL_MÉDULLAIRE (écrase GLUTEUS si chevauchement)

    # Sauvegarder sous forme d’image PNG à 1 canal
    Image.fromarray(fused, mode='L').save(os.path.join(output_dir, fname))

print("✅ Fusion terminée avec succès.")


### 🔹Étape 5 : Entraînement du modèle U-Net multiclasses 🎯


Après avoir préparé nos images (rognage, annotation, fusion des masques, redimensionnement), on lance l’entraînement du modèle de segmentation.

On utilise un U-Net multiclasses pour détecter trois zones : fond (0), muscle GLUTEUS (1) et canal médullaire (2).

Pour bien apprendre, on crée un générateur de données qui charge les images et leurs masques, et applique des augmentations aléatoires (retournements, rotations, changements de luminosité…) pour rendre le modèle plus robuste face aux variations.

Le réseau est construit en plusieurs couches convolutionnelles, avec des connexions “skip” qui aident à garder les détails. On utilise aussi du Dropout pour éviter le surapprentissage.

Le modèle est entraîné avec l’optimiseur Adam et la perte categorical_crossentropy, qui convient bien à la segmentation multiclasses. Des callbacks comme EarlyStopping et ReduceLROnPlateau permettent d’arrêter l’entraînement quand il n’y a plus de progrès et d’ajuster le taux d’apprentissage.

En résumé, ce modèle donne un bon équilibre entre qualité de segmentation et temps d’entraînement, et il pourra être amélioré plus tard avec des architectures plus complexes ou des pertes adaptées.

Le modèle final est sauvegardé pour être utilisé ensuite en détection automatique sur les images de carcasses. 🚀

#### CODE A EXECUTER SOUS GOOGLE COLLAB

In [ ]:
import os
import numpy as np
import tensorflow as tf
import albumentations as A
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, BatchNormalization, Activation, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard

# ---------------------------
# Paramètres d'images et d'entraînement
# ---------------------------
IMG_HEIGHT, IMG_WIDTH = 512, 512
BATCH_SIZE = 16
EPOCHS = 150
NUM_CLASSES = 3  # classes: 0,1,2

# ---------------------------
# Chemins vers les dossiers contenant les images et les masques multiclasses
# ---------------------------
image_dir = '/content/IMG'
mask_dir = '/content/MSK'  # nouveaux masques à valeurs {0,1,2}

# ---------------------------
# Data Augmentation avec Albumentations
# ---------------------------
def get_augmentations():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(p=0.2)
    ])

# ---------------------------
# DataGenerator multiclasses
# ---------------------------
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_dir, mask_dir, batch_size, augmentations=None):
        self.image_paths = [os.path.join(image_dir, f)
                            for f in os.listdir(image_dir)
                            if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        self.mask_paths = [os.path.join(mask_dir, os.path.basename(f))
                           for f in self.image_paths]
        self.batch_size = batch_size
        self.augment = augmentations
        self.indexes = np.arange(len(self.image_paths))

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, idx):
        batch_indexes = self.indexes[idx*self.batch_size:(idx+1)*self.batch_size]
        images = []
        masks = []
        for i in batch_indexes:
            # Lecture et normalisation de l'image
            img = load_img(self.image_paths[i], target_size=(IMG_HEIGHT, IMG_WIDTH))
            img = img_to_array(img) / 255.0

            # Lecture du masque multiclasses
            mask = load_img(self.mask_paths[i], color_mode='grayscale', target_size=(IMG_HEIGHT, IMG_WIDTH))
            mask = img_to_array(mask)[..., 0].astype(np.int32)  # shape (H, W)

            # Application des augmentations
            if self.augment:
                augmented = self.augment(image=img, mask=mask)
                img = augmented['image']
                mask = augmented['mask']

            # One-hot encoding du masque: shape (H, W, NUM_CLASSES)
            mask_cat = tf.keras.utils.to_categorical(mask, num_classes=NUM_CLASSES)

            images.append(img)
            masks.append(mask_cat)

        return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

    def on_epoch_end(self):
        np.random.shuffle(self.indexes)

# Instantiate generators
train_gen = DataGenerator(image_dir, mask_dir, batch_size=BATCH_SIZE, augmentations=get_augmentations())
val_gen = DataGenerator(image_dir, mask_dir, batch_size=BATCH_SIZE, augmentations=None)

# ---------------------------
# Bloc de convolution (U-Net)
# ---------------------------
def conv_block(input_tensor, num_filters):
    x = Conv2D(num_filters, (3, 3), kernel_initializer='he_normal', padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(num_filters, (3, 3), kernel_initializer='he_normal', padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

# ---------------------------
# Construction du U-Net multiclasses
# ---------------------------
def unet_multiclass(input_size=(IMG_HEIGHT, IMG_WIDTH, 3), num_classes=NUM_CLASSES):
    inputs = Input(input_size)

    c1 = conv_block(inputs, 32)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = conv_block(p1, 64)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = conv_block(p2, 128)
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = conv_block(p3, 256)
    p4 = MaxPooling2D((2, 2))(c4)

    c42 = conv_block(p4, 512)
    p42 = MaxPooling2D((2, 2))(c4)

    c5 = conv_block(p42, 1024)
    c5 = Dropout(0.5)(c5)

    u6 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c5)
    u6 = concatenate([u6, c4])
    c6 = conv_block(u6, 256)

    u7 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = concatenate([u7, c3])
    c7 = conv_block(u7, 128)

    u8 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = concatenate([u8, c2])
    c8 = conv_block(u8, 64)

    u9 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = concatenate([u9, c1])
    c9 = conv_block(u9, 32)

    # Sortie multiclasses
    outputs = Conv2D(num_classes, (1, 1), activation='softmax')(c9)

    model = Model(inputs, outputs)
    return model

model = unet_multiclass()

# ---------------------------
# Compilation: perte et métriques
# ---------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ---------------------------
# Callbacks
# ---------------------------
checkpoint_path = '/content/best_model_multiclass.keras'
callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, mode='min', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    TensorBoard(log_dir='logs_multiclass', histogram_freq=1)
]

# ---------------------------
# Entraînement
# ---------------------------
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks
)

# Chargement du meilleur modèle et sauvegarde finale
model.load_weights(checkpoint_path)
model.save('/content/final_model_multiclass.keras')


### 🔹Étape 6 : Conversion du modèle en TFLite pour Raspberry Pi 5 🚀


Pour accélérer l’inférence sur Raspberry Pi 5 et gagner en efficacité, on convertit le modèle Keras en format TensorFlow Lite (TFLite).

Ce premier script fait une conversion sans quantification, donc le modèle reste en float32 (précision complète). C’est simple et rapide pour commencer.

Le script charge le modèle Keras sauvegardé, le convertit en TFLite, et sauvegarde le fichier .tflite prêt à être déployé sur le RPi.

👉 Pas de perte de précision, mais le modèle est un peu plus lourd et moins optimisé que la version quantifiée.

Tu pourras ensuite appliquer la quantification INT-8 pour encore plus de performance et moins d’espace, je te montrerai ça juste après. 😉

In [ ]:
import tensorflow as tf
import jax

def convert_keras_to_tflite_no_quant(keras_model_path: str, tflite_model_path: str) -> None:
    print(f"Chargement du modèle Keras depuis : {keras_model_path}")
    model = tf.keras.models.load_model(keras_model_path, compile=False)

    # Conversion sans optimisation (prend la précision float32 d'origine)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    print("Conversion en TFLite sans quantification en cours...")
    try:
        tflite_model = converter.convert()
    except Exception as e:
        print("La conversion a échoué :", e)
        return

    with open(tflite_model_path, "wb") as f:
        f.write(tflite_model)

    print(f"Conversion réussie, modèle sauvegardé sous : {tflite_model_path}")

if __name__ == "__main__":
    keras_model_path = "/content/TETS.keras"
    tflite_model_path = "/content/OK.tflite"
    convert_keras_to_tflite_no_quant(keras_model_path, tflite_model_path)


### Conversion quantifiée INT-8 pour TFLite 🔥
Pour optimiser encore plus la vitesse et la taille du modèle sur Raspberry Pi 5, on utilise la quantification INT-8.

Ce script :

Charge ton modèle Keras

Utilise un petit jeu de calibration (quelques images) pour ajuster la quantification

Force le modèle à utiliser des opérations INT-8 pour un maximum d’efficacité

Sauvegarde le modèle quantifié prêt à être déployé

👍 Résultat : un modèle plus léger et ultra-rapide, parfait pour l’embarqué !

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# -----------------------------------------
# Paramètres
# -----------------------------------------
IMG_HEIGHT, IMG_WIDTH = 512, 512  # ou 512 si tu veux, mais 256 est plus rapide sur RPi
IMAGE_DIR = '/content/VAL'  # Chemin vers tes images d'entraînement
KERAS_MODEL_PATH = '/content/TETS.keras'
TFLITE_MODEL_PATH = '/content/DOUBLE_QUINTOA_512.tflite'

# -----------------------------------------
# Representative dataset generator
# -----------------------------------------
def representative_data_gen():
    image_files = [
        os.path.join(IMAGE_DIR, f)
        for f in os.listdir(IMAGE_DIR)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]
    print(f"Found {len(image_files)} images for calibration.")

    # En général 50-100 suffisent
    for img_path in image_files[:100]:
        img = load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
        img = img_to_array(img) / 255.0
        img = np.expand_dims(img.astype(np.float32), axis=0)
        yield [img]

# -----------------------------------------
# Conversion
# -----------------------------------------
def convert_keras_to_tflite_int8(keras_model_path, tflite_model_path):
    print(f"Chargement du modèle Keras depuis : {keras_model_path}")
    model = tf.keras.models.load_model(keras_model_path, compile=False)

    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # 1️⃣ Activer l'optimisation par défaut (quantization-aware)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # 2️⃣ Fournir le jeu de calibration
    converter.representative_dataset = representative_data_gen

    # 3️⃣ Forcer INT8 pur
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8

    # 4️⃣ Convertir
    print("Conversion en TFLite INT8 en cours...")
    try:
        tflite_model = converter.convert()
    except Exception as e:
        print("La conversion a échoué :", e)
        return

    # 5️⃣ Sauvegarder
    with open(tflite_model_path, "wb") as f:
        f.write(tflite_model)

    print(f"Conversion réussie ! Modèle quantifié INT8 sauvegardé sous : {tflite_model_path}")

# -----------------------------------------
# Entrée du script
# -----------------------------------------
if __name__ == "__main__":
    convert_keras_to_tflite_int8(KERAS_MODEL_PATH, TFLITE_MODEL_PATH)


# Le modèle est prêt à l’emploi 🚀, optimisé et efficace pour une utilisation directe sur Raspberry Pi.
